# Utilização da extensão 'Parquet' - Transformando arquivos 'xlsx' em 'parquet': Criado pelo Nicholas Birochi

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Impportando Bibliotecas
from pathlib import Path        # Salvando no endereço certo...
from glob import glob           # Busca de arquivos por padrões...
import pandas as pd             # Análise e manipulação de dados...
import os                       # Operações com arquivos e sistema...

#### Criando as funções necessárias para esse processo dos ativos das pessoas:

In [ ]:
# Função para remover as duplicatas entre as abas:
def remover_duplicados_por_nome_cadastro(df_base, df_outra,
                                         col_id="Cadastro",
                                         col_nome="Nome completo"):
    """
    Remove do df_base todas as linhas cujo (Cadastro, Nome completo)
    também exista em df_outra.
    """
    # Garante que as colunas existem nas duas bases
    if col_id not in df_base.columns or col_nome not in df_base.columns:
        return df_base
    if col_id not in df_outra.columns or col_nome not in df_outra.columns:
        return df_base

    def normaliza(s):
        return s.astype(str).str.strip().str.upper()

    # Normaliza as chaves nas duas bases
    chave_base = normaliza(df_base[col_id]) + "||" + normaliza(df_base[col_nome])
    chave_outra = normaliza(df_outra[col_id]) + "||" + normaliza(df_outra[col_nome])

    conjunto_outra = set(chave_outra.unique())

    # Mantém só quem NÃO está na outra base
    mask = ~chave_base.isin(conjunto_outra)
    df_filtrado = df_base[mask].copy()

    return df_filtrado


# Função para a conversão de cada uma das colunas em formato "string", assim não perdendo nenhum dado:
def converter_colunas_mistas_para_str(df):
    for col in df.columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

def ler_sem_colunas(path, colunas_excluir, **kwargs):
    colunas = pd.read_excel(path, nrows=0, **kwargs).columns.tolist()

    if all(isinstance(c, int) for c in colunas_excluir):
        colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
    else:
        colunas_ler = [col for col in colunas if col not in colunas_excluir]

    return pd.read_excel(path, usecols=colunas_ler, **kwargs)

# Função final para a transformação de cada arquivo:
def converter_xlsx_para_parquet(pasta_xlsx, colunas_excluir=None):
    # Define o caminho de saída:
    pasta_parquet = R'..\..\..\Base de Dados Original\parquet\Ativos'
    
    # Buscando cada arquivo em formato ".xlsx":
    arquivos_xlsx = glob(os.path.join(pasta_xlsx, "*.xlsx"))
    
    # Procurando os arquivos ".xlsx" e usando das funções para trocar para ".parquet"
    for caminho in arquivos_xlsx:
        try:
            # Lê a aba de DEPARTAMENTOS (normalmente sheet 0)
            if colunas_excluir:
                df = ler_sem_colunas(caminho, colunas_excluir)
            else:
                df = pd.read_excel(caminho, dtype=str)

            # 🔹 Lê a outra aba (por ex., AFASTADOS) para poder comparar
            try:
                df_outra = pd.read_excel(caminho, sheet_name=1, dtype=str)
            except Exception:
                df_outra = pd.DataFrame()  # se não existir, não filtra nada

            # 🔹 Remove do df (ativos) quem também está na outra aba
            if not df_outra.empty:
                df = remover_duplicados_por_nome_cadastro(
                    df_base=df,
                    df_outra=df_outra,
                    col_id="Cadastro",
                    col_nome="Nome completo"
                )

            # Corrigindo coluna específica:
            if 'Movimento de Colaboradores' in df.columns:
                df['Movimento de Colaboradores'] = df['Movimento de Colaboradores'].astype(str)
            
            # Convertendo todas as colunas em string:
            df = converter_colunas_mistas_para_str(df)
            
            # Pegando o nome original do arquivo sem extensão:
            nome_arquivo = f"Ativos.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            # Salvando o arquivo:
            df.to_parquet(caminho_salvar, index=False, engine='pyarrow')
            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")

#### Inserindo os endereços e nomes para dentro das funções para o processamento dos ativos das pessoas:

In [ ]:
# Uso para a base de Ativos:
converter_xlsx_para_parquet(
    pasta_xlsx=R'..\..\..\Base de Dados Original\xlsx\Ativos e Desligados',
    colunas_excluir=[4, 6, 7, 9, 12, 13, 16, 17]
)

#### Criando as funções necessárias para o processamento das pessoas as quais foram desligadas:

In [ ]:
# Função para converter colunas com tipos mistos para string
def converter_colunas_mistas_para_str(df):
    for col in df.columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

# Função para ler o Excel ignorando as colunas que deseja excluir
def ler_sem_colunas(path, colunas_excluir=None, sheet_name=1, **kwargs):
    # Pega todas as colunas da planilha (sheet_name)
    colunas = pd.read_excel(path, sheet_name=sheet_name, nrows=0, **kwargs).columns.tolist()

    if colunas_excluir:
        if all(isinstance(c, int) for c in colunas_excluir):
            # Exclui colunas por índice
            colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
        else:
            # Exclui colunas por nome
            colunas_ler = [col for col in colunas if col not in colunas_excluir]
    else:
        colunas_ler = colunas

    # Lê só as colunas filtradas e a planilha desejada
    return pd.read_excel(path, sheet_name=sheet_name, usecols=colunas_ler, **kwargs)

# Função principal que converte todos os XLSX da pasta para parquet
def converter_xlsx_para_parquet(pasta_xlsx, colunas_excluir=None):
    # Pasta onde será salvo o parquet (crie/ajuste o caminho)
    pasta_parquet = R'..\..\..\Base de Dados Original\parquet\Desligados'

    # Cria a pasta se não existir
    os.makedirs(pasta_parquet, exist_ok=True)

    # Lista todos os arquivos xlsx na pasta
    arquivos_xlsx = glob(os.path.join(pasta_xlsx, "*.xlsx"))

    for caminho in arquivos_xlsx:
        try:
            if colunas_excluir:
                df = ler_sem_colunas(caminho, colunas_excluir=colunas_excluir, sheet_name=1, dtype=str)
            else:
                df = pd.read_excel(caminho, sheet_name=1, dtype=str)

            # Caso tenha coluna específica para converter
            if 'Movimento de Colaboradores' in df.columns:
                df['Movimento de Colaboradores'] = df['Movimento de Colaboradores'].astype(str)

            # Converte colunas com tipos mistos para string
            df = converter_colunas_mistas_para_str(df)

            # Define nome do arquivo parquet
            nome_arquivo = f"Desligados.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            # Salva o dataframe como parquet
            df.to_parquet(caminho_salvar, index=False, engine='pyarrow')
            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")

#### Inserindo os endereços e nomes para dentro das funções para o processamento das pessoas as quais foram desligadas:

In [ ]:
# Uso para a base de Desligados:
converter_xlsx_para_parquet(
    pasta_xlsx=R'..\..\..\Base de Dados Original\xlsx\Ativos e Desligados',
    colunas_excluir=[2, 4, 5, 6, 9, 10]
)

#### Criando as funções necessárias para o processamento das informações de dentro do GPTW:

In [ ]:
# Função para converter os tipos das colunas em string, para não perder informações:
def converter_colunas_mistas_para_str(df):
    for col in df.columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

# Função para ler o Excel ignorando as colunas que deseja excluir
def ler_sem_colunas(path, colunas_excluir=None, sheet_name=0, **kwargs):
    # Pega todas as colunas da planilha (sheet_name)
    colunas = pd.read_excel(path, sheet_name=sheet_name, nrows=0, **kwargs).columns.tolist()

    if colunas_excluir:
        if all(isinstance(c, int) for c in colunas_excluir):
            # Exclui colunas por índice
            colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
        else:
            # Exclui colunas por nome
            colunas_ler = [col for col in colunas if col not in colunas_excluir]
    else:
        colunas_ler = colunas

    # Lê só as colunas filtradas e a planilha desejada
    return pd.read_excel(path, sheet_name=sheet_name, usecols=colunas_ler, **kwargs)

# Função principal que converte todos os XLSX da pasta para parquet
def converter_xlsx_para_parquet(pasta_xlsx, colunas_excluir=None):
    # Pasta onde será salvo o parquet (crie/ajuste o caminho)
    pasta_parquet = r'..\..\..\Base de Dados Original\parquet\GPTW'

    # Cria a pasta se não existir
    os.makedirs(pasta_parquet, exist_ok=True)

    # Lista todos os arquivos xlsx na pasta
    arquivos_xlsx = glob(os.path.join(pasta_xlsx, "*.xlsx"))

    for caminho in arquivos_xlsx:
        try:
            if colunas_excluir:
                df = ler_sem_colunas(caminho, colunas_excluir=colunas_excluir, sheet_name=0, dtype=str)
            else:
                df = pd.read_excel(caminho, sheet_name=0, dtype=str)

            # Caso tenha coluna específica para converter
            if 'Movimento de Colaboradores' in df.columns:
                df['Movimento de Colaboradores'] = df['Movimento de Colaboradores'].astype(str)

            # Converte colunas com tipos mistos para string
            df = converter_colunas_mistas_para_str(df)

            # Define nome do arquivo parquet
            nome_arquivo = f"GPTW.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            # Salva o dataframe como parquet
            df.to_parquet(caminho_salvar, index=False, engine='pyarrow')
            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")

#### Inserindo todas as colunas das informações sobre o GPTW:

In [ ]:
# Uso para a base da Tabela GPTW:
converter_xlsx_para_parquet(
    pasta_xlsx=R'..\..\..\Base de Dados Original\xlsx\GPTW',
    colunas_excluir=[4, 5]
)

#### Criando as funções necessárias para o processamento das informações de dentro das Movimentações internas e externas da empresa::

In [ ]:
# Função para converter os tipos das colunas em string, para não perder informações:
def converter_colunas_mistas_para_str(df, preservar=None):
    preservar = set(preservar or [])
    for col in df.columns:
        if col not in preservar and df[col].map(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

def ler_sem_colunas(path, colunas_excluir=None, sheet_name=0, **kwargs):
    cols = pd.read_excel(path, sheet_name=sheet_name, nrows=0, **kwargs).columns.tolist()
    if colunas_excluir:
        if all(isinstance(c, int) for c in colunas_excluir):
            usecols = [c for i, c in enumerate(cols) if i not in colunas_excluir]
        else:
            usecols = [c for c in cols if c not in colunas_excluir]
    else:
        usecols = cols
    return pd.read_excel(path, sheet_name=sheet_name, usecols=usecols, **kwargs)

def _pick_col(df, *names):
    u = {c.strip().upper(): c for c in df.columns}
    ups = [n.strip().upper() for n in names if n]
    for n in ups:
        if n in u:
            return u[n]
    for c in df.columns:
        cu = c.strip().upper()
        if any(n in cu for n in ups):
            return c
    return None

# pipeline compacto
def converter_xlsx_para_parquet(
    pasta_xlsx: str,
    colunas_excluir=None,
    inicio: str = '2023-01-01',
    fim: str | None = None,
    pasta_parquet: str | None = None,
    verbose: bool = True
):
    p = Path(pasta_xlsx).resolve()
    if pasta_parquet is None:
        pasta_parquet = r'..\..\..\Base de Dados Original\parquet\Movimentacoes'

    # >>> ALTERAÇÃO: garantir que a pasta de saída exista
    Path(pasta_parquet).mkdir(parents=True, exist_ok=True)

    arquivos = sorted(Path(pasta_xlsx).glob("*.xlsx"))
    if not arquivos and verbose:
        print(f"⚠️ Nenhum .xlsx em: {pasta_xlsx}")

    dt_inicio = pd.to_datetime(inicio, format="%Y-%m-%d")
    dt_fim = pd.to_datetime(fim, format="%Y-%m-%d") if fim else None

    for arq in arquivos:
        try:
            df = ler_sem_colunas(arq, colunas_excluir=colunas_excluir, sheet_name=0, dtype=str)
            n0 = len(df)

            # coluna de data
            col_data = _pick_col(df, "Data de alteração")
            if not col_data:
                raise ValueError("Coluna de data não encontrada.")

            # parse data (dd/mm/aaaa, etc. + fallback serial excel)
            raw = df[col_data].astype(str)
            parsed = raw.str.extract(r'(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})', expand=False)
            s = pd.to_datetime(parsed, errors="coerce", dayfirst=True)
            if s.isna().all():
                s = pd.to_datetime(raw, errors="coerce", dayfirst=True)
            if s.isna().mean() > 0.4:
                as_num = pd.to_numeric(raw.str.replace(",", ".", regex=False), errors="coerce")
                s = s.fillna(pd.to_datetime(as_num, errors="coerce", unit="D", origin="1899-12-30"))
            df[col_data] = s

            mask = (df[col_data] >= dt_inicio) & ((df[col_data] <= dt_fim) if dt_fim is not None else True)
            df = df.loc[mask].reset_index(drop=True)

            df = converter_colunas_mistas_para_str(df, preservar=[col_data])

            # Define nome do arquivo parquet
            nome_arquivo = "Movimentacoes.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            # >>> ALTERAÇÃO: salvar de verdade o parquet
            df.to_parquet(caminho_salvar, index=False)

            if verbose:
                kept = len(df)
                print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"❌ {arq.name}: {e}")


#### Inserindo todas as colunas das informações sobre as Movimentações internas e externas da empresa:

In [ ]:
# Uso para a base de Movimentações:
converter_xlsx_para_parquet(
    pasta_xlsx=r'..\..\..\Base de Dados Original\xlsx\Movimentacoes',
    colunas_excluir=[1,2,3,4,6,9,10,12,15,16,19],
    inicio='2023-01-01',
    fim='2025-09-30'
)